In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  


In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
BRONZE_PATH_SELIC  = "workspace.case_spark_cvm.bronze_selic_anual"
BRONZE_PATH_CDI    = "workspace.case_spark_cvm.bronze_cdi_diario"
BRONZE_PATH_IPCA   = "workspace.case_spark_cvm.bronze_ipca_mensal"
BRONZE_PATH_IBOV   = "workspace.case_spark_cvm.bronze_ibov_index"

NOME_TABELA  = f"silver_dados_indicadores_economicos" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## Valores - Indicador Desempenho

### 1.1 tratemento silver

#### 1.1.1 LEITURA DOS DADOS BRONZE

In [0]:
df_bronze_selic = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH_SELIC, partition_col="data_processamento" )

df_bronze_cdi = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH_CDI, partition_col="data_processamento" )

df_bronze_ipca = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH_IPCA, partition_col="data_processamento" )

df_bronze_ibov = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH_IBOV, partition_col="data_processamento" )

#### 1.1.2 TRATAMENTO SELIC, CDI E IBOV

In [0]:
df_selic = df_bronze_selic\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,2)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_selic")


df_cdi = df_bronze_cdi\
    .withColumn(
    "data",
    f.date_format(f.to_date(f.col("data"), "dd/MM/yyyy"), "yyyy-MM-dd")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("valor", "valor_cdi")


df_ibov = df_bronze_ibov\
    .withColumn(
    "data",
        f.from_unixtime(f.col("timestamp")).cast("date")
    )\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("close", f.col("close").cast(t.DecimalType(20,6)))\
    .withColumn("data_processamento", f.col("data_processamento").cast(t.IntegerType()))\
    .withColumnRenamed("close", "ibov_close")\
    .drop("timestamp")\
    .select("data", "ibov_close", "_source_url", "_ingest_timestamp",  "data_processamento" )


df_ipca = df_bronze_ipca\
    .withColumn("data", f.date_format(f.to_date(f.col("data"), "dd/MM/yyyy"), "yyyy-MM-dd"))\
    .withColumn("data", f.col("data").cast(t.DateType()))\
    .withColumn("valor", f.col("valor").cast(t.DecimalType(10,4)))\
    .filter(f.col("data").isNotNull())\
    .withColumnRenamed("valor", "ipca_mensal")\
    .withColumnRenamed("data", "data_ipca")

#### 1.1.3 Retirando dados duplicados

In [0]:
def processar_indicador_silver(df, chave_indicador, coluna_valor, nome_tabela_origem):
    # 1. Espalha o valor oficial
    window_data = Window.partitionBy(chave_indicador)
    df_processado = df.withColumn(
        "_valor_oficial",
        f.max(
            f.when(f.col("_ingest_timestamp") == f.max("_ingest_timestamp").over(window_data), f.col(coluna_valor))
        ).over(window_data)
    )

    # 2. Reaproveita a função do config
    df_silver, df_duplicadas = PipelineConfig.remover_duplicatas(
        df=df_processado,
        chave_negocio=chave_indicador,
        coluna_ordenacao="_ingest_timestamp"
    )

    # 3. Quarentena Inteligente
    df_quarentena = (df_duplicadas
        .filter(f.col(coluna_valor) != f.col("_valor_oficial"))
        .withColumn("_motivo_quarentena", f.lit("Anomalia API: A fonte alterou retroativamente o valor deste indicador financeiro"))
    )

    # 4. Limpeza
    df_silver = df_silver.drop("_valor_oficial")
    df_quarentena = df_quarentena.drop("_valor_oficial")

    # 5. Salva a Quarentena
    PipelineConfig.salvar_quarentena(
        spark=spark,
        df_quarentena=df_quarentena, 
        tabela_origem=nome_tabela_origem, 
        data_proc=DATA_PROC
    )
    
    return df_silver



df_silver_selic = processar_indicador_silver(df_selic, ["data"], "valor_selic", "bronze_selic_anual")
df_silver_cdi   = processar_indicador_silver(df_cdi, ["data"], "valor_cdi", "bronze_cdi_diario")
df_silver_ibov  = processar_indicador_silver(df_ibov, ["data"], "ibov_close", "bronze_ibov_index")
df_silver_ipca  = processar_indicador_silver(df_ipca, ["data_ipca"], "ipca_mensal", "bronze_ipca_mensal")


# Dropando as colunas de metadados
df_silver_selic = df_silver_selic.drop("_source_url", "_ingest_timestamp", "data_processamento")
df_silver_cdi   = df_silver_cdi.drop("_source_url", "_ingest_timestamp", "data_processamento")
df_silver_ibov  = df_silver_ibov.drop("_source_url", "_ingest_timestamp", "data_processamento")
df_silver_ipca  = df_silver_ipca.drop("_source_url", "_ingest_timestamp", "data_processamento")


#### 1.1.3 TRATAMENTO IPCA (MENSAL) E CÁLCULO DO ACUMULADO (12 MESES)

In [0]:
# Para calcular o IPCA acumulado de 12 meses corretamente (juros compostos)
# Fator = 1 + (ipca_mensal / 100)
df_silver_ipca = df_silver_ipca.withColumn("fator", (f.col("ipca_mensal") / 100) + 1).coalesce(1)


# Usamos uma Window para pegar os últimos 12 meses ordenados pela data
# Acumulado = (Produto dos fatores de 12 meses) - 1. No PySpark: EXP(SUM(LOG(fator)))
window_12m = Window.orderBy("data_ipca").rowsBetween(-11, Window.currentRow)


# Janela Histórica para o Índice Acumulado Base
window_ipca_historico = Window.orderBy("data_ipca").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_ipca = df_silver_ipca \
    .withColumn("fator_acumulado", f.exp(f.sum(f.log("fator")).over(window_12m))) \
    .withColumn("ipca_anual", ((f.col("fator_acumulado") - 1) * 100).cast(t.DecimalType(10, 2)))\
    .withColumn("indice_ipca", f.exp(f.sum(f.log("fator")).over(window_ipca_historico))) # CRIAÇÃO DO ÍNDICE

# Criamos uma chave Ano-Mês para facilitar o join com os dados diários
df_silver_ipca = df_silver_ipca \
    .withColumn("ano_mes", f.date_format("data_ipca", "yyyy-MM")) \
    .select("ano_mes", "ipca_mensal", "ipca_anual", "indice_ipca")

#### 1.1.4 CRIAÇÃO DE UM CALENDÁRIO ÚNICO E JOIN DOS INDICADORES

In [0]:
# Extraímos todas as datas únicas disponíveis entre Selic e CDI

df_datas = (df_silver_selic.select("data")
            .union(df_silver_cdi.select("data"))
            .union(df_silver_ibov.select("data"))
            .distinct()
            )

# Criamos a chave Ano-Mês nas datas base
df_datas = df_datas.withColumn("ano_mes", f.date_format("data", "yyyy-MM"))

# Realizamos o Join: left com Selic, left com CDI, left com IPCA
df_indicadores = df_datas \
    .join(df_silver_selic, "data", "left") \
    .join(df_silver_cdi, "data", "left") \
    .join(df_silver_ibov, "data", "left")\
    .join(df_silver_ipca, "ano_mes", "left")\
    .coalesce(1)




#### 1.1.5 CONTORNO DO PROBLEMA DE IPCA ATRASADO (FORWARD FILL) E AJUSTANDO OS INDICES (SELIC/IPCA/CDI)

In [0]:
# Para os dias cujos meses ainda não têm IPCA lançado (Ex: fev/mar de 2026 ficarão nulos no join),
# preenchemos com o último valor de IPCA conhecido usando a função last() com ignorenulls=True.
window_ffill = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)
# Criação dos Índices Acumulados Diários para Selic e CDI
window_historico_diario = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_indicadores = df_indicadores \
    .withColumn("ipca_mensal", f.last("ipca_mensal", ignorenulls=True).over(window_ffill)) \
    .withColumn("ipca_anual", f.last("ipca_anual", ignorenulls=True).over(window_ffill))\
    .withColumn("indice_ipca", f.last("indice_ipca", ignorenulls=True).over(window_ffill))


df_indicadores = df_indicadores \
    .withColumn("fator_cdi", 1 + (f.coalesce(f.col("valor_cdi"), f.lit(0)) / 100)) \
    .withColumn("fator_selic", 
        # Desanualizando a Selic (elevando a 1/252)
        f.when(f.col("valor_selic").isNotNull(),
               f.pow(1 + (f.col("valor_selic") / 100), 1/252)
        ).otherwise(f.lit(1)) # Se for nulo/fim de semana, fator é 1 (rende zero)       
    ) \
    .withColumn("indice_cdi", f.exp(f.sum(f.log("fator_cdi")).over(window_historico_diario))) \
    .withColumn("indice_selic", f.exp(f.sum(f.log("fator_selic")).over(window_historico_diario)))

# Limpamos a tabela para o formato final e adicionamos data_processamento
data_proc = int(datetime.now().strftime("%Y%m%d"))

df_silver_indicadores = df_indicadores \
    .select(
        "data", 
        "valor_selic", 
        "valor_cdi", 
        "ipca_mensal", 
        "ipca_anual", 
        "ibov_close",
        "indice_cdi",   
        "indice_selic",  
        "indice_ipca"    
    )



#### 1.1.6 TRATAMENTO DO TIPO DE DADO

In [0]:

df_silver_indicadores = df_silver_indicadores.select(
    f.col('data').cast(t.DateType()).alias('data'),
    f.col('valor_selic').cast(t.DecimalType(10, 4)).alias('valor_selic'),
    f.col('valor_cdi').cast(t.DecimalType(10, 6)).alias('valor_cdi'),
    f.col('ipca_mensal').cast(t.DecimalType(10, 4)).alias('ipca_mensal'),
    f.col('ipca_anual').cast(t.DecimalType(10, 4)).alias('ipca_anual'),
    f.col('ibov_close').cast(t.DecimalType(18, 2)).alias('ibov_close'),
    f.col('indice_cdi').cast(t.DecimalType(20, 8)).alias('indice_cdi'),
    f.col('indice_selic').cast(t.DecimalType(20, 8)).alias('indice_selic'),
    f.col('indice_ipca').cast(t.DecimalType(20, 8)).alias('indice_ipca')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["data"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_indicadores, 
    tabela_destino= SILVER_PATH, 
    chave_negocio=chave_negocio
    )